# **CV融合算子开发**
本章将通过代码辅助讲解CV融合算子的基础理论，以及CV融合算子的实现：
- 环境准备
- CV融合算子分析
- CV融合算子实现
- CV融合算子测试
- 课后实践
---

## **1. 环境准备**
本文所有内容均存放于Sources文件夹。
在开始创建算子工程前，先要对jupyter环境进行初始化。以下代码完成了初始化并将环境中的变量导入jupyter环境，并完成代码目录的创建。保证能正常导入代码以及编译。

In [ ]:
!mkdir -p Sources/03.05

import os
import subprocess
from pathlib import Path

set_env = os.environ.get("ASCEND_TOOLKIT_HOME", "/usr/local/Ascend/cann") + "/set_env.sh"
if not Path(set_env).exists():
    set_env = "/usr/local/Ascend/cann/set_env.sh"

result = subprocess.run(
    ["bash", "-lc", f"source {set_env} && env"],
    capture_output=True,
    text=True,
    check=True,
)
for line in result.stdout.strip().split("\n"):
    if "=" in line and not line.startswith(("#", " ")):
        key, value = line.split("=", 1)
        os.environ[key] = value

print("Environment initialization process completed successfully.")

---
## **2. CV融合算子分析**
假设我们有个简单的小模型，里面包含了一个Matmul、一个Leakyrelu算子，原始的模型如下：  

<img src="./images/03.05/cv_operator_model.png">

### **2.1 算子输入输出与Shape信息**
融合后的算子仅包含输入 `a`、`b`、`bias` 和输出 `c`，其计算流程为：

$$
c = leakyrelu(a \times b + bias, alpha)
$$

其中 `a`、`b` 为 `half` 类型输入，`bias` 为 `float` 类型输入，`c` 为 `float` 类型输出。此处仅以 `M = 1024`，`K = 256`，`N = 640` 输入数据作为演示示例，在真实业务场景下，可灵活调整输入数据的shape参数，也可直接支持泛化输入形式。

<div style="text-align: left; float: left;">
<table style="border-collapse: collapse; width: 100%; max-width: 650px; border: 1px solid #ddd;">
  <tr>
    <td colspan="1" style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">算子名</td>
    <td colspan="4" style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;text-align: center;" >matmul_leakyrelu_custom</td>
  </tr>
  <tr>
    <td rowspan="4" style="padding: 8px; border-right: 1px solid #ddd; font-weight: bold; background-color: #f5f5f5; vertical-align: middle;">算子输入</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">name</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">shape</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">data type</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">format</td>
  </tr>
  <tr>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">a</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">(M, K) / (1024, 256)</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">half</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">ND</td>
  </tr>
  <tr>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">b</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">(K, N) / (256, 640)</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">half</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">ND</td>
  </tr>
  <tr>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">bias</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">(N) / (640)</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">float</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">ND</td>
  </tr>
  <tr>
    <td style="padding: 8px; border-right: 1px solid #ddd; font-weight: bold; background-color: #f5f5f5; vertical-align: middle;">算子输出</td>
    <td style="padding: 8px;">c</td>
    <td style="padding: 8px;">(M, N) / (1024, 640)</td>
    <td style="padding: 8px;">float</td>
    <td style="padding: 8px;">ND</td>
  </tr>
</table>
</div>
<div style="clear: both;"></div>


### **2.2 执行步骤拆解**
Matmul与LeakyReLU两个算子原生采用串行执行模式，完整流程可拆解为 **6 个独立且串行的步骤**：
- 阶段1（Matmul执行）：Matmul输入数据搬入 → Cube核执行Matmul计算 → Matmul输出数据搬出  

- 阶段2（LeakyReLU执行）：LeakyReLU输入数据搬入 → Vector核执行 LeakyReLU计算 → LeakyReLU输出数据搬出

### **2.3 核心效率瓶颈**
- **计算核资源利用率低**：Matmul依赖Cube核运算，LeakyReLU依赖Vector核运算，串行执行时总有一类计算核处于完全空闲状态；  

- **数据传输开销大**：LeakyReLU输入完全依赖Matmul输出，中间结果需经历“全局内存 ↔ 计算核”的两次完整搬移，额外增加访存耗时；  

- **执行链路冗长**：6 个独立步骤串行执行，无并行优化空间，端到端耗时高。  


将Matmul与LeakyReLU融合为单一算子后，通过「数据切块+核间协同+本地缓存」重构执行流程，整体以 **5 个核心步骤循环执行** 完成全量数据运算：

<div style="text-align: left; float: left;">
<table style="border-collapse: collapse; width: 100%; max-width: 1485px; border: 1px solid #ddd;">
  <tr>
    <td style="width: 60px; padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">步骤</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">操作内容</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">核心目的</td>
  </tr>
  <tr>
    <td style="width: 60px; padding: 8px; border-bottom: 1px solid #ddd;">1</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">数据切块：将Matmul输入数据（a矩阵、b矩阵、bias偏置）按 Cube 核单次计算能力切分为若干数据块</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">Cube每运算完成一小块数据可以直接让Vector核开始计算，避免Vector核空闲等待</td>
  </tr>
  <tr>
    <td style="width: 60px; padding: 8px; border-bottom: 1px solid #ddd;">2</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">Cube核数据搬入：将单块输入数据直接搬入Cube核本地存储</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">数据需要放在Cube核上完成矩阵乘运算</td>
  </tr>
  <tr>
    <td style="width: 60px; padding: 8px; border-bottom: 1px solid #ddd;">3</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">Cube核并行计算Matmul：各Cube核并行完成对应数据块的Matmul计算，结果暂存于LocalTensor</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">利用Matmul API的能力，直接输出Matmul计算结果到UB，减少编写搬运代码</td>
  </tr>
  <tr>
    <td style="width: 60px; padding: 8px; border-bottom: 1px solid #ddd;">4</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">Vector核并行计算LeakyReLU：直接读取LocalTensor中的Matmul结果，完成LeakyReLU 计算并暂存</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">核间协同无数据搬移，Vector核无空闲</td>
  </tr>
  <tr>
    <td style="width: 60px; padding: 8px;">5</td>
    <td style="padding: 8px;">结果统一搬出：将LeakyReLU结果从LocalTensor搬至全局内存指定偏移地址</td>
    <td style="padding: 8px;">将计算结果放在正确的位置</td>
  </tr>
</table>
</div>
<div style="clear: both;"></div>

### **2.4 融合核心优化效果**
- **算力利用率提升**：Cube/Vector 核在同一算子调用中协同工作，大幅减少核空闲时长，提升硬件利用率；

---
## **3. CV融合算子host侧tiling实现**
Matmul在Cube上的数据流向为：GM →L1→L0A/L0B →Cube →L0C→FixPipe→GM  
LeakRelu在Cube上的数据流向为：GM → UB → Vector → UB → GM
所以，在Host侧实现时，需要考虑每一次计算的数据小块大小不能超出L0C的大小和UB的大小。这里以A2设备展示：  
 
<img src="./images/03.05/a2_frame.png" alt="turing_test"  width="700px">  

Host 侧调用 `MultiCoreMatmulTiling` 生成 `AscendC::tiling::TCubeTiling` 结构体，并作为Kernel参数直接传入Device侧。本示例中 Host 侧 `numBlocks = 1`，Kernel 使用 `__mix__(1, 2)`，对应1个AIC与2个AIV；`SetDim(2)` 配置参与 Matmul 发起的AIV数量。

In [ ]:
%%writefile Sources/03.05/matmul_leakyrelu_custom.asc
#include <algorithm>
#include <cstdint>
#include <cstdio>
#include <vector>

#include "acl/acl.h"
#include "kernel_operator.h"
#include "kernel_tiling/kernel_tiling.h"
#include "lib/matmul_intf.h"
#include "tiling/platform/platform_ascendc.h"
#include "tiling/tiling_api.h"

using namespace matmul;

#define CHECK_ACL(expr)                                                                                 \
    do {                                                                                                \
        auto __ret = (expr);                                                                            \
        int32_t __code = static_cast<int32_t>(__ret);                                                   \
        if (__code != 0) {                                                                              \
            std::fprintf(stderr, "[ERROR] %s failed at %s:%d, ret=%d\n", #expr, __FILE__, __LINE__,    \
                         __code);                                                                       \
            return 1;                                                                                   \
        }                                                                                               \
    } while (0)

AscendC::tiling::TCubeTiling GenerateTiling(platform_ascendc::PlatformAscendC* ascendcPlatform)
{
    constexpr int32_t M = 1024;
    constexpr int32_t N = 640;
    constexpr int32_t K = 256;
    constexpr int32_t baseM = 128;
    constexpr int32_t baseN = 128;

    matmul_tiling::MultiCoreMatmulTiling cubeTiling(*ascendcPlatform);
    cubeTiling.SetDim(2);
    cubeTiling.SetAType(matmul_tiling::TPosition::GM, matmul_tiling::CubeFormat::ND,
                        matmul_tiling::DataType::DT_FLOAT16);
    cubeTiling.SetBType(matmul_tiling::TPosition::GM, matmul_tiling::CubeFormat::ND,
                        matmul_tiling::DataType::DT_FLOAT16);
    cubeTiling.SetCType(matmul_tiling::TPosition::VECIN, matmul_tiling::CubeFormat::ND,
                        matmul_tiling::DataType::DT_FLOAT);
    cubeTiling.SetBiasType(matmul_tiling::TPosition::GM, matmul_tiling::CubeFormat::ND,
                           matmul_tiling::DataType::DT_FLOAT);
    cubeTiling.SetShape(M, N, K);
    cubeTiling.SetOrgShape(M, N, K);
    cubeTiling.SetFixSplit(baseM, baseN, -1);
    cubeTiling.SetBias(true);
    cubeTiling.SetTraverse(matmul_tiling::MatrixTraverse::FIRSTM);
    cubeTiling.SetBufferSpace(-1, -1, -1);

    AscendC::tiling::TCubeTiling cubeTilingData;
    if (cubeTiling.GetTiling(cubeTilingData) == -1) {
        std::fprintf(stderr, "Generate matmul tiling failed.\n");
    }
    return cubeTilingData;
}

__aicore__ inline uint32_t Ceiling(uint32_t a, uint32_t b)
{
    return (a + b - 1) / b;
}



---
## **4. CV融合算子kernel侧代码实现**
`matmul_leakyrelu_custom.asc` 中完成CV融合算子的Kernel实现开发。
MatmulLeakyRelu融合算子的整体计算流程为：Matmul计算数据搬入->Matmul计算->Matmul计算结果搬出->LeakyRelu计算数据搬入->LeakyRelu计算->LeakyRelu计算结果搬出。  

<img src="./images/03.05/data_trends.png"  alt="turing_test"  width="700px">    

其中Matmul计算数据搬入、Matmul计算结果搬出和LeakyRelu计算数据搬入的操作可以通过Matmul API直接实现。 所以我们可以定义算子类包含以下函数：  
- MatmulCompute : 实现Matmul计算逻辑。
- LeakyReluCompute : 实现LeakyRelu计算逻辑。
- CopyOut : LeakyRelu计算结果搬出操作。CopyOut函数的实现逻辑与Vector算子运算结果的搬出流程不同。核心原因在于MatmulLeakyRelu算子的输出是一小块矩阵，导致其向GM的数据写入无法采用直接填充的方式。具体操作要求为：每完成一行小矩阵计算结果的填充后，需先跳过(整体矩阵单行元素数 - 小矩阵单行元素数)的地址偏移量，再执行下一行小矩阵结果的填充操作。    
    <img src="./images/03.05/result.png"  alt="turing_test"  width="450px">   
   
- CalcOffset: 计算偏移函数。    
    1. **分块方式**：左矩阵（A）按行维度分块，右矩阵（B）按列维度分块，单个核负责的计算任务为「左矩阵某行块 × 右矩阵某列块」，运算结果对应结果矩阵（C）中对应的子矩阵区域。  

    2. **核任务分配逻辑（以示例说明）**：  
        若左矩阵分为 A1、A2 两个行块，右矩阵分为 B1、B2 两个列块，共启用 4 个核运算，则按 M 方向遍历规则分配任务：先按核编号依次遍历左矩阵所有行块，完成一轮遍历后，右矩阵列块向右滑动一列进入下一轮。具体分配为：  
        Core0：A1 × B1  
        Core1：A2 × B1  
        Core2：A1 × B2  
        Core3：A2 × B2   
        <img src="./images/03.05/offset.png"  alt="turing_test"  width="400px">     

    3. **分块索引计算**：  
    当前核处理的左矩阵行块 = 核编号 % 左矩阵总块数  
    当前核处理的右矩阵列块 = 核编号 / 左矩阵总块数（整除）  
    A 偏移 = 左块索引 × 单块行数 × A 列数（转置则为左块索引 × 单块行数）  
    B 偏移 = 右块索引 × 单块列数（转置则为右块索引 × 单块列数 × B 行数）  
    C 偏移 = 左块起始行 × C 总列数 + 右块起始列  

- Init：根据CalcOffset函数计算得到当前核处理的A矩阵偏移、B矩阵偏移、C矩阵偏移。  
- Process：设置当前核处理的A矩阵、B矩阵、C矩阵指针，并通过Iterate获取每次运算得到的C矩阵小块结果，用于LeakyRelu计算。

In [ ]:
%%writefile -a Sources/03.05/matmul_leakyrelu_custom.asc

template <typename AType, typename BType, typename CType, typename BiasType>
class MatmulLeakyKernel {
public:
    __aicore__ inline void Init(__gm__ uint8_t* a, __gm__ uint8_t* b, __gm__ uint8_t* bias, __gm__ uint8_t* c,
                                const TCubeTiling& tiling, float alpha, AscendC::TPipe* pipe);
    __aicore__ inline void Process();
    __aicore__ inline void MatmulCompute();
    __aicore__ inline void LeakyReluCompute();
    __aicore__ inline void CopyOut(uint32_t count);
    __aicore__ inline void CalcOffset(int32_t blockIdx, const TCubeTiling& tiling, int32_t& offsetA,
                                      int32_t& offsetB, int32_t& offsetC, int32_t& offsetBias);

    Matmul<MatmulType<AscendC::TPosition::GM, CubeFormat::ND, AType>,
           MatmulType<AscendC::TPosition::GM, CubeFormat::ND, BType>,
           MatmulType<AscendC::TPosition::VECIN, CubeFormat::ND, CType>,
           MatmulType<AscendC::TPosition::GM, CubeFormat::ND, BiasType>>
        matmulObj;

    AscendC::GlobalTensor<AType> aGlobal;
    AscendC::GlobalTensor<BType> bGlobal;
    AscendC::GlobalTensor<CType> cGlobal;
    AscendC::GlobalTensor<BiasType> biasGlobal;
    AscendC::LocalTensor<CType> reluOutLocal;
    TCubeTiling tiling;
    float alpha = 0.001f;
    AscendC::TQue<AscendC::QuePosition::VECOUT, 1> reluOutQueue;
};

template <typename AType, typename BType, typename CType, typename BiasType>
__aicore__ inline void MatmulLeakyKernel<AType, BType, CType, BiasType>::Init(
    __gm__ uint8_t* a, __gm__ uint8_t* b, __gm__ uint8_t* bias, __gm__ uint8_t* c, const TCubeTiling& tiling,
    float alpha, AscendC::TPipe* pipe)
{
    this->tiling = tiling;
    this->alpha = alpha;
    aGlobal.SetGlobalBuffer(reinterpret_cast<__gm__ AType*>(a), tiling.M * tiling.Ka);
    bGlobal.SetGlobalBuffer(reinterpret_cast<__gm__ BType*>(b), tiling.Kb * tiling.N);
    cGlobal.SetGlobalBuffer(reinterpret_cast<__gm__ CType*>(c), tiling.M * tiling.N);
    biasGlobal.SetGlobalBuffer(reinterpret_cast<__gm__ BiasType*>(bias), tiling.N);

    int32_t offsetA = 0;
    int32_t offsetB = 0;
    int32_t offsetC = 0;
    int32_t offsetBias = 0;
    CalcOffset(AscendC::GetBlockIdx(), tiling, offsetA, offsetB, offsetC, offsetBias);
    aGlobal = aGlobal[offsetA];
    bGlobal = bGlobal[offsetB];
    cGlobal = cGlobal[offsetC];
    biasGlobal = biasGlobal[offsetBias];
    pipe->InitBuffer(reluOutQueue, 1, tiling.baseM * tiling.baseN * sizeof(CType));
}

template <typename AType, typename BType, typename CType, typename BiasType>
__aicore__ inline void MatmulLeakyKernel<AType, BType, CType, BiasType>::Process()
{
    uint32_t computeRound = 0;
    matmulObj.SetTensorA(aGlobal);
    matmulObj.SetTensorB(bGlobal);
    matmulObj.SetBias(biasGlobal);
    while (matmulObj.template Iterate<true>()) {
        MatmulCompute();
        LeakyReluCompute();
        CopyOut(computeRound);
        ++computeRound;
    }
    matmulObj.End();
}

template <typename AType, typename BType, typename CType, typename BiasType>
__aicore__ inline void MatmulLeakyKernel<AType, BType, CType, BiasType>::MatmulCompute()
{
    reluOutLocal = reluOutQueue.AllocTensor<CType>();
    matmulObj.template GetTensorC<true>(reluOutLocal, false, true);
}

template <typename AType, typename BType, typename CType, typename BiasType>
__aicore__ inline void MatmulLeakyKernel<AType, BType, CType, BiasType>::LeakyReluCompute()
{
    AscendC::LeakyRelu(reluOutLocal, reluOutLocal, static_cast<CType>(alpha), tiling.baseM * tiling.baseN);
    reluOutQueue.EnQue(reluOutLocal);
}

template <typename AType, typename BType, typename CType, typename BiasType>
__aicore__ inline void MatmulLeakyKernel<AType, BType, CType, BiasType>::CopyOut(uint32_t count)
{
    reluOutQueue.DeQue<CType>();
    const uint32_t roundM = tiling.singleCoreM / tiling.baseM;
    uint32_t startOffset = count % roundM * tiling.baseM * tiling.N + count / roundM * tiling.baseN;
    AscendC::DataCopyParams copyParam = {
        static_cast<uint16_t>(tiling.baseM),
        static_cast<uint16_t>(tiling.baseN * sizeof(CType) / AscendC::DEFAULT_C0_SIZE),
        0,
        static_cast<uint16_t>((tiling.N - tiling.baseN) * sizeof(CType) / AscendC::DEFAULT_C0_SIZE)};
    AscendC::DataCopy(cGlobal[startOffset], reluOutLocal, copyParam);
    reluOutQueue.FreeTensor(reluOutLocal);
}

template <typename AType, typename BType, typename CType, typename BiasType>
__aicore__ inline void MatmulLeakyKernel<AType, BType, CType, BiasType>::CalcOffset(
    int32_t blockIdx, const TCubeTiling& tiling, int32_t& offsetA, int32_t& offsetB, int32_t& offsetC,
    int32_t& offsetBias)
{
    uint32_t mSingleBlocks = Ceiling(tiling.M, tiling.singleCoreM);
    uint32_t mCoreIdx = blockIdx % mSingleBlocks;
    uint32_t nCoreIdx = blockIdx / mSingleBlocks;
    offsetA = mCoreIdx * tiling.Ka * tiling.singleCoreM;
    offsetB = nCoreIdx * tiling.singleCoreN;
    offsetC = mCoreIdx * tiling.N * tiling.singleCoreM + nCoreIdx * tiling.singleCoreN;
    offsetBias = nCoreIdx * tiling.singleCoreN;
}

// 通过高阶API开发的CV融合算子，需要通过__kfc_workspace__标识workspace，由编译器完成workspace初始化。 
extern "C" __global__ __mix__(1, 2) void matmul_leakyrelu_custom(
    __gm__ uint8_t* a, __gm__ uint8_t* b, __gm__ uint8_t* bias, __gm__ uint8_t* c,
    __gm__ uint8_t* __kfc_workspace__ workspace, AscendC::tiling::TCubeTiling tiling)
{
    AscendC::TPipe pipe;

    MatmulLeakyKernel<half, half, float, float> op;
    op.Init(a, b, bias, c, tiling, 0.001f, &pipe);
    REGIST_MATMUL_OBJ(&pipe, GetSysWorkSpacePtr(), op.matmulObj, &op.tiling);
    op.Process();
}


---
## **5. CV融合算子测试**
完成Host侧tiling、Kernel实现和Host调用代码后，在同一个 `.asc` 文件中追加 `main` 函数。这里以固定的 M = 1024，N = 640，K = 256 进行测试，a矩阵全1，b矩阵全2，bias全0.5。

In [ ]:
%%writefile -a Sources/03.05/matmul_leakyrelu_custom.asc

int32_t main(int32_t argc, char** argv)
{
    auto ascendcPlatform = platform_ascendc::PlatformAscendCManager::GetInstance();

    constexpr uint32_t M = 1024;
    constexpr uint32_t N = 640;
    constexpr uint32_t K = 256;
    constexpr uint32_t numBlocks = 1;
    constexpr size_t aFileSize = M * K * sizeof(aclFloat16);
    constexpr size_t bFileSize = K * N * sizeof(aclFloat16);
    constexpr size_t biasFileSize = N * sizeof(float);
    constexpr size_t cFileSize = M * N * sizeof(float);
    const size_t workspaceSize = static_cast<size_t>(ascendcPlatform->GetLibApiWorkSpaceSize());
    auto tiling = GenerateTiling(ascendcPlatform);

    constexpr int32_t deviceId = 0;
    aclrtStream stream = nullptr;
    CHECK_ACL(aclInit(nullptr));
    CHECK_ACL(aclrtSetDevice(deviceId));
    CHECK_ACL(aclrtCreateStream(&stream));

    uint8_t* inputADevice = nullptr;
    uint8_t* inputBDevice = nullptr;
    uint8_t* inputBiasDevice = nullptr;
    uint8_t* outputCDevice = nullptr;
    uint8_t* workspaceDevice = nullptr;
    CHECK_ACL(aclrtMalloc(reinterpret_cast<void**>(&inputADevice), aFileSize, ACL_MEM_MALLOC_HUGE_FIRST));
    CHECK_ACL(aclrtMalloc(reinterpret_cast<void**>(&inputBDevice), bFileSize, ACL_MEM_MALLOC_HUGE_FIRST));
    CHECK_ACL(aclrtMalloc(reinterpret_cast<void**>(&inputBiasDevice), biasFileSize, ACL_MEM_MALLOC_HUGE_FIRST));
    CHECK_ACL(aclrtMalloc(reinterpret_cast<void**>(&outputCDevice), cFileSize, ACL_MEM_MALLOC_HUGE_FIRST));
    if (workspaceSize > 0) {
        CHECK_ACL(aclrtMalloc(reinterpret_cast<void**>(&workspaceDevice), workspaceSize, ACL_MEM_MALLOC_HUGE_FIRST));
    }

    std::vector<aclFloat16> inputAHost(M * K, aclFloatToFloat16(1.0f));
    std::vector<aclFloat16> inputBHost(K * N, aclFloatToFloat16(2.0f));
    std::vector<float> inputBiasHost(N, 0.5f);
    std::vector<float> outputCHost(M * N, 0.0f);
    std::vector<float> golden(M * N, 512.5f);

    CHECK_ACL(aclrtMemcpy(inputADevice, aFileSize, inputAHost.data(), aFileSize, ACL_MEMCPY_HOST_TO_DEVICE));
    CHECK_ACL(aclrtMemcpy(inputBDevice, bFileSize, inputBHost.data(), bFileSize, ACL_MEMCPY_HOST_TO_DEVICE));
    CHECK_ACL(aclrtMemcpy(inputBiasDevice, biasFileSize, inputBiasHost.data(), biasFileSize,
                          ACL_MEMCPY_HOST_TO_DEVICE));

    matmul_leakyrelu_custom<<<numBlocks, 0, stream>>>(inputADevice, inputBDevice, inputBiasDevice, outputCDevice,
                                                       workspaceDevice, tiling);
    CHECK_ACL(aclrtSynchronizeStream(stream));
    CHECK_ACL(aclrtMemcpy(outputCHost.data(), cFileSize, outputCDevice, cFileSize, ACL_MEMCPY_DEVICE_TO_HOST));

    std::printf("result is:\n");
    for (uint32_t i = 0; i < 10; ++i) {
        std::printf("%.1f ", outputCHost[i]);
    }
    std::printf("\ntest %s\n", std::equal(outputCHost.begin(), outputCHost.end(), golden.begin()) ? "pass" : "failed");

    CHECK_ACL(aclrtFree(inputADevice));
    CHECK_ACL(aclrtFree(inputBDevice));
    CHECK_ACL(aclrtFree(inputBiasDevice));
    CHECK_ACL(aclrtFree(outputCDevice));
    if (workspaceDevice != nullptr) {
        CHECK_ACL(aclrtFree(workspaceDevice));
    }
    CHECK_ACL(aclrtDestroyStream(stream));
    CHECK_ACL(aclrtResetDevice(deviceId));
    CHECK_ACL(aclFinalize());
    return 0;
}


完成全部 `.asc` 代码后，还需要配置CMake来调用Ascend C编译工具链。由于本样例在Host侧调用 `MultiCoreMatmulTiling` 生成Matmul Tiling，并且调用 `PlatformAscendCManager` 获取硬件平台信息，因此还需要额外链接 `tiling_api`、`register`、`platform` 等CANN库。此外通过 `TARGET_SRC` 变量指定待编译的 `.asc` 源文件，方便课后实践复用同一份配置。

In [ ]:
%%writefile Sources/03.05/CMakeLists.txt
cmake_minimum_required(VERSION 3.16)

set(CMAKE_ASC_RUN_MODE "npu" CACHE STRING "Run mode: npu, cpu, sim")
set(CMAKE_ASC_ARCHITECTURES "dav-2201" CACHE STRING "NPU architecture: dav-2201, dav-3510")
set(TARGET_SRC "matmul_leakyrelu_custom.asc" CACHE STRING "ASC source file used to build demo")

find_package(ASC REQUIRED)

project(kernel_samples LANGUAGES ASC CXX)

add_executable(demo
    ${TARGET_SRC}
)

target_link_libraries(demo PRIVATE
    tiling_api
    register
    platform
    m
    dl
)

target_compile_options(demo PRIVATE
    $<$<COMPILE_LANGUAGE:ASC>:--npu-arch=${CMAKE_ASC_ARCHITECTURES}>
)


编译直调工程，检查 `.asc` 文件能生成可执行程序。

In [ ]:
!cd Sources/03.05 && \
mkdir -p build && \
cd build && \
cmake -DCMAKE_ASC_ARCHITECTURES=dav-2201 .. && \
make -j

运行测试程序，检查输出是否与预期一致。

In [ ]:
!cd Sources/03.05/build && ./demo

调用成功后，会有如下输出：
```
result is:
512.5 512.5 512.5 512.5 512.5 512.5 512.5 512.5 512.5 512.5 
test pass
```

---
# **课后实践**
尝试修改代码，开发一个新的CV融合算子，实现 Matmul + Bias + Abs 的计算。公式为：
<div style="width: 20%; float: left; clear: both; margin: 10px 0;">

$$
C = \text{abs}(A \times B + Bias)
$$
</div>
<div style="clear: both;"></div>

支持M = 1024, N = 640, K = 256的half类型输入，float类型输出，算子原型为：

<div style="text-align: left; float: left;">
<table style="border-collapse: collapse; width: 100%; max-width: 550px; border: 1px solid #ddd;">
  <tr>
    <td colspan="1" style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">算子名</td>
    <td colspan="4" style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;text-align: center;" >matmul_abs</td>
  </tr>
  <tr>
    <td rowspan="4" style="padding: 8px; border-right: 1px solid #ddd; font-weight: bold; background-color: #f5f5f5; vertical-align: middle;">算子输入</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">name</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">shape</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">data type</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">format</td>
  </tr>
  <tr>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">a</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">(1024, 256)</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">half</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">ND</td>
  </tr>
  <tr>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">b</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">(256, 640)</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">half</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">ND</td>
  </tr>
   <tr>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">bias</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">(640)</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">float</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">ND</td>
  </tr> 
  <tr>
  <td  style="padding: 8px; border-right: 1px solid #ddd; font-weight: bold; background-color: #f5f5f5; vertical-align: middle;">算子输出</td>
    <td style="padding: 8px;">c</td>
    <td style="padding: 8px;">(1024, 640)</td>
    <td style="padding: 8px;">float</td>
    <td style="padding: 8px;">ND</td>
  </tr>
</table>
</div>
<div style="clear: both;"></div>  

### **编写MatmulAbs直调算子**
参照课程中 `matmul_leakyrelu_custom.asc` 的工程形式，完成 `matmul_abs.asc`。下方代码仅保留直调工程框架、数据准备和函数骨架，`TODO` 处需要学习者补全：Tiling配置、Matmul结果获取、Abs融合计算、结果搬出和分块偏移计算。


In [ ]:
%%writefile Sources/03.05/matmul_abs.asc
#include <algorithm>
#include <cstdint>
#include <cstdio>
#include <vector>

#include "acl/acl.h"
#include "kernel_operator.h"
#include "kernel_tiling/kernel_tiling.h"
#include "lib/matmul_intf.h"
#include "tiling/platform/platform_ascendc.h"
#include "tiling/tiling_api.h"

using namespace matmul;

#define CHECK_ACL(expr)                                                                                 \
    do {                                                                                                \
        auto __ret = (expr);                                                                            \
        int32_t __code = static_cast<int32_t>(__ret);                                                   \
        if (__code != 0) {                                                                              \
            std::fprintf(stderr, "[ERROR] %s failed at %s:%d, ret=%d\n", #expr, __FILE__, __LINE__,    \
                         __code);                                                                       \
            return 1;                                                                                   \
        }                                                                                               \
    } while (0)

AscendC::tiling::TCubeTiling GenerateTiling(platform_ascendc::PlatformAscendC* ascendcPlatform)
{
    constexpr int32_t M = 1024;
    constexpr int32_t N = 640;
    constexpr int32_t K = 256;
    constexpr int32_t baseM = 128;
    constexpr int32_t baseN = 128;

    matmul_tiling::MultiCoreMatmulTiling cubeTiling(*ascendcPlatform);

    // TODO: 参照课程示例配置Matmul Tiling参数。
    // 需要完成的内容包括：SetDim、A/B/C/Bias类型、Shape、OrgShape、FixSplit、Bias、Traverse和BufferSpace。

    AscendC::tiling::TCubeTiling cubeTilingData;
    if (cubeTiling.GetTiling(cubeTilingData) == -1) {
        std::fprintf(stderr, "Generate matmul tiling failed.\n");
    }
    return cubeTilingData;
}

__aicore__ inline uint32_t Ceiling(uint32_t a, uint32_t b)
{
    return (a + b - 1) / b;
}


template <typename AType, typename BType, typename CType, typename BiasType>
class MatmulAbsKernel {
public:
    __aicore__ inline void Init(__gm__ uint8_t* a, __gm__ uint8_t* b, __gm__ uint8_t* bias, __gm__ uint8_t* c,
                                const TCubeTiling& tiling, AscendC::TPipe* pipe);
    __aicore__ inline void Process();
    __aicore__ inline void MatmulCompute();
    __aicore__ inline void AbsCompute();
    __aicore__ inline void CopyOut(uint32_t count);
    __aicore__ inline void CalcOffset(int32_t blockIdx, const TCubeTiling& tiling, int32_t& offsetA,
                                      int32_t& offsetB, int32_t& offsetC, int32_t& offsetBias);

    Matmul<MatmulType<AscendC::TPosition::GM, CubeFormat::ND, AType>,
           MatmulType<AscendC::TPosition::GM, CubeFormat::ND, BType>,
           MatmulType<AscendC::TPosition::VECIN, CubeFormat::ND, CType>,
           MatmulType<AscendC::TPosition::GM, CubeFormat::ND, BiasType>>
        matmulObj;

    AscendC::GlobalTensor<AType> aGlobal;
    AscendC::GlobalTensor<BType> bGlobal;
    AscendC::GlobalTensor<CType> cGlobal;
    AscendC::GlobalTensor<BiasType> biasGlobal;
    AscendC::LocalTensor<CType> absOutLocal;
    TCubeTiling tiling;
    AscendC::TQue<AscendC::QuePosition::VECOUT, 1> absOutQueue;
};

template <typename AType, typename BType, typename CType, typename BiasType>
__aicore__ inline void MatmulAbsKernel<AType, BType, CType, BiasType>::Init(
    __gm__ uint8_t* a, __gm__ uint8_t* b, __gm__ uint8_t* bias, __gm__ uint8_t* c, const TCubeTiling& tiling,
    AscendC::TPipe* pipe)
{
    this->tiling = tiling;

    // TODO: 设置aGlobal、bGlobal、cGlobal、biasGlobal的GM地址和元素个数。
    // TODO: 调用CalcOffset计算当前逻辑核处理的A/B/C/Bias偏移，并更新GlobalTensor起始地址。
    // TODO: 初始化absOutQueue缓存。
}

template <typename AType, typename BType, typename CType, typename BiasType>
__aicore__ inline void MatmulAbsKernel<AType, BType, CType, BiasType>::Process()
{
    // TODO: 绑定当前核的A/B/Bias输入。
    // TODO: 通过Iterate循环获取每个baseM * baseN的Matmul结果，依次调用MatmulCompute、AbsCompute和CopyOut。
    // TODO: 结束Matmul对象。
}

template <typename AType, typename BType, typename CType, typename BiasType>
__aicore__ inline void MatmulAbsKernel<AType, BType, CType, BiasType>::MatmulCompute()
{
    // TODO: 从absOutQueue申请LocalTensor，并通过GetTensorC获取Matmul中间结果。
}

template <typename AType, typename BType, typename CType, typename BiasType>
__aicore__ inline void MatmulAbsKernel<AType, BType, CType, BiasType>::AbsCompute()
{
    // TODO: 对Matmul结果执行AscendC::Abs，并将结果EnQue到absOutQueue。
}

template <typename AType, typename BType, typename CType, typename BiasType>
__aicore__ inline void MatmulAbsKernel<AType, BType, CType, BiasType>::CopyOut(uint32_t count)
{
    // TODO: 从absOutQueue取出Abs结果。
    // TODO: 按FIRSTM遍历顺序计算当前输出小块写回C矩阵的startOffset。
    // TODO: 使用DataCopyParams和DataCopy将小块结果搬出到GM。
    // TODO: 释放LocalTensor。
}

template <typename AType, typename BType, typename CType, typename BiasType>
__aicore__ inline void MatmulAbsKernel<AType, BType, CType, BiasType>::CalcOffset(
    int32_t blockIdx, const TCubeTiling& tiling, int32_t& offsetA, int32_t& offsetB, int32_t& offsetC,
    int32_t& offsetBias)
{
    // TODO: 根据blockIdx、singleCoreM、singleCoreN计算当前逻辑核对应的M/N方向分块索引。
    // TODO: 根据分块索引计算A、B、C、Bias在GM上的起始偏移。
}

extern "C" __global__ __mix__(1, 2) void matmul_abs(
    __gm__ uint8_t* a, __gm__ uint8_t* b, __gm__ uint8_t* bias, __gm__ uint8_t* c,
    __gm__ uint8_t* __kfc_workspace__ workspace, AscendC::tiling::TCubeTiling tiling)
{
    AscendC::TPipe pipe;

    MatmulAbsKernel<half, half, float, float> op;
    op.Init(a, b, bias, c, tiling, &pipe);
    REGIST_MATMUL_OBJ(&pipe, GetSysWorkSpacePtr(), op.matmulObj, &op.tiling);
    op.Process();
}

int32_t main(int32_t argc, char** argv)
{
    auto ascendcPlatform = platform_ascendc::PlatformAscendCManager::GetInstance();

    constexpr uint32_t M = 1024;
    constexpr uint32_t N = 640;
    constexpr uint32_t K = 256;
    constexpr uint32_t numBlocks = 1;
    constexpr size_t aFileSize = M * K * sizeof(aclFloat16);
    constexpr size_t bFileSize = K * N * sizeof(aclFloat16);
    constexpr size_t biasFileSize = N * sizeof(float);
    constexpr size_t cFileSize = M * N * sizeof(float);
    const size_t workspaceSize = static_cast<size_t>(ascendcPlatform->GetLibApiWorkSpaceSize());
    auto tiling = GenerateTiling(ascendcPlatform);

    constexpr int32_t deviceId = 0;
    aclrtStream stream = nullptr;
    CHECK_ACL(aclInit(nullptr));
    CHECK_ACL(aclrtSetDevice(deviceId));
    CHECK_ACL(aclrtCreateStream(&stream));

    uint8_t* inputADevice = nullptr;
    uint8_t* inputBDevice = nullptr;
    uint8_t* inputBiasDevice = nullptr;
    uint8_t* outputCDevice = nullptr;
    uint8_t* workspaceDevice = nullptr;
    CHECK_ACL(aclrtMalloc(reinterpret_cast<void**>(&inputADevice), aFileSize, ACL_MEM_MALLOC_HUGE_FIRST));
    CHECK_ACL(aclrtMalloc(reinterpret_cast<void**>(&inputBDevice), bFileSize, ACL_MEM_MALLOC_HUGE_FIRST));
    CHECK_ACL(aclrtMalloc(reinterpret_cast<void**>(&inputBiasDevice), biasFileSize, ACL_MEM_MALLOC_HUGE_FIRST));
    CHECK_ACL(aclrtMalloc(reinterpret_cast<void**>(&outputCDevice), cFileSize, ACL_MEM_MALLOC_HUGE_FIRST));
    if (workspaceSize > 0) {
        CHECK_ACL(aclrtMalloc(reinterpret_cast<void**>(&workspaceDevice), workspaceSize, ACL_MEM_MALLOC_HUGE_FIRST));
    }

    std::vector<aclFloat16> inputAHost(M * K, aclFloatToFloat16(-1.0f));
    std::vector<aclFloat16> inputBHost(K * N, aclFloatToFloat16(2.0f));
    std::vector<float> inputBiasHost(N, 0.5f);
    std::vector<float> outputCHost(M * N, 0.0f);
    std::vector<float> golden(M * N, 511.5f);

    CHECK_ACL(aclrtMemcpy(inputADevice, aFileSize, inputAHost.data(), aFileSize, ACL_MEMCPY_HOST_TO_DEVICE));
    CHECK_ACL(aclrtMemcpy(inputBDevice, bFileSize, inputBHost.data(), bFileSize, ACL_MEMCPY_HOST_TO_DEVICE));
    CHECK_ACL(aclrtMemcpy(inputBiasDevice, biasFileSize, inputBiasHost.data(), biasFileSize,
                          ACL_MEMCPY_HOST_TO_DEVICE));

    matmul_abs<<<numBlocks, 0, stream>>>(inputADevice, inputBDevice, inputBiasDevice, outputCDevice,
                                         workspaceDevice, tiling);
    CHECK_ACL(aclrtSynchronizeStream(stream));
    CHECK_ACL(aclrtMemcpy(outputCHost.data(), cFileSize, outputCDevice, cFileSize, ACL_MEMCPY_DEVICE_TO_HOST));

    std::printf("result is:\n");
    for (uint32_t i = 0; i < 10; ++i) {
        std::printf("%.1f ", outputCHost[i]);
    }
    std::printf("\ntest %s\n", std::equal(outputCHost.begin(), outputCHost.end(), golden.begin()) ? "pass" : "failed");

    CHECK_ACL(aclrtFree(inputADevice));
    CHECK_ACL(aclrtFree(inputBDevice));
    CHECK_ACL(aclrtFree(inputBiasDevice));
    CHECK_ACL(aclrtFree(outputCDevice));
    if (workspaceDevice != nullptr) {
        CHECK_ACL(aclrtFree(workspaceDevice));
    }
    CHECK_ACL(aclrtDestroyStream(stream));
    CHECK_ACL(aclrtResetDevice(deviceId));
    CHECK_ACL(aclFinalize());
    return 0;
}


### **编译运行**
补全 `matmul_abs.asc` 中的 TODO 后，使用同一个 `CMakeLists.txt`，通过 `TARGET_SRC=matmul_abs.asc` 指定课后实践源码文件进行编译运行。


In [ ]:
!cd Sources/03.05 && \
mkdir -p build_abs && \
cd build_abs && \
cmake -DTARGET_SRC=matmul_abs.asc -DCMAKE_ASC_ARCHITECTURES=dav-2201 .. && \
make -j && \
./demo

预期输出：
```
result is:
511.5 511.5 511.5 511.5 511.5 511.5 511.5 511.5 511.5 511.5 
test pass
```

**执行以下命令查看答案**  

matmul_abs.asc

In [ ]:
!cat ./answer/03.05/matmul_abs.asc